In [1]:
# %% [markdown]
# ### Phase 1: Setup and Directory Initialization

import os
import pandas as pd
import numpy as np

# Define our directory structures
RAW_DATA_DIR = "data"
PROCESSED_DATA_DIR = "data/processed"

# Create the processed directory if it doesn't exist yet
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
print(f"📦 Environment ready. Standardized outputs will be saved to: {PROCESSED_DATA_DIR}")

📦 Environment ready. Standardized outputs will be saved to: data/processed


In [4]:
# %% [markdown]
# ### Phase 2: Standardizing Training Set (GSE122488) - FIXED HEADERS

# 1. Define file path
gse122488_path = os.path.join(RAW_DATA_DIR, "1_training_GSE122488", "GSE122488_normalized_microRNA_counts.txt")

print("⏳ Loading and cleaning GSE122488 with corrected headers...")

# 2. Load dataset - telling pandas row 3 holds the column headers (sample IDs)
df_train = pd.read_csv(gse122488_path, sep="\t", header=3, index_col=0)

# 3. Keep ONLY rows where the microRNAs start with 'hsa-' 
df_train = df_train[df_train.index.str.startswith('hsa-', na=False)]

# Clean up the index name for a flawless table format
df_train.index.name = "miRNA"

# 4. Save the flawless matrix to our processed folder
output_path_train = os.path.join(PROCESSED_DATA_DIR, "GSE122488_expression_clean.csv")
df_train.to_csv(output_path_train)

print("✅ GSE122488 Processing Complete!")
print(f"   -> Final Shape (miRNAs x Samples): {df_train.shape}")
print(f"   -> First 3 verified sample IDs: {list(df_train.columns[:3])}")
print(f"   -> Saved to: {output_path_train}\n")

# Display a quick preview of the beautifully structured dataframe
df_train.iloc[:5, :4]

⏳ Loading and cleaning GSE122488 with corrected headers...
✅ GSE122488 Processing Complete!
   -> Final Shape (miRNAs x Samples): (1939, 38)
   -> First 3 verified sample IDs: ['HC1', 'HC2', 'HC3']
   -> Saved to: data/processed\GSE122488_expression_clean.csv



,HC1,HC2,HC3,HC4
miRNA,,,,
hsa-let-7a-2-3p,0.0,0.0,0.0,0.0
hsa-let-7a-3p,229.9,94.9,196.7,198.3
hsa-let-7a-5p,59698.6,81394.0,46040.3,100482.5
hsa-let-7b-3p,31.5,28.9,16.5,23.8
hsa-let-7b-5p,11130.6,17069.5,8196.9,30959.1


In [5]:
# %% [markdown]
# ### Phase 3: Standardizing Independent Validation Set (GSE184472)

# 1. Define file path (Update extension to .xlsx if you saved it as such)
gse184472_path = os.path.join(RAW_DATA_DIR, "2_validation_GSE184472", "GSE184472_Processed_data_GBMvsNormal.xls")

print("⏳ Loading and prefix-correcting GSE184472 (Excel format)...")

# 2. Load Excel dataset
df_val = pd.read_excel(gse184472_path, index_col=0)

# 3. Add the missing 'hsa-' prefix to align names with our Training and CGGA datasets
df_val.index = "hsa-" + df_val.index.astype(str)

# Clean up the index name for complete alignment
df_val.index.name = "miRNA"

# 4. Save the corrected validation matrix to our processed folder
output_path_val = os.path.join(PROCESSED_DATA_DIR, "GSE184472_expression_clean.csv")
df_val.to_csv(output_path_val)

print("✅ GSE184472 Processing Complete!")
print(f"   -> Final Shape (miRNAs x Samples): {df_val.shape}")
print(f"   -> Corrected First 3 miRNAs: {list(df_val.index[:3])}")
print(f"   -> Saved to: {output_path_val}\n")

# Preview the top-left section of our newly formatted table
df_val.iloc[:5, :4]

⏳ Loading and prefix-correcting GSE184472 (Excel format)...
✅ GSE184472 Processing Complete!
   -> Final Shape (miRNAs x Samples): (2083, 118)
   -> Corrected First 3 miRNAs: ['hsa-let-7a-2-3p', 'hsa-let-7a-3p', 'hsa-let-7a-5p']
   -> Saved to: data/processed\GSE184472_expression_clean.csv



,Normal-P01,Normal-P02,Normal-P03,Normal-P04
miRNA,,,,
hsa-let-7a-2-3p,19,30,21,34
hsa-let-7a-3p,14,3,16,17
hsa-let-7a-5p,871,343,1224,1460
hsa-let-7b-5p,851,456,979,920
hsa-let-7c-3p,1,4,11,11


In [6]:
# %% [markdown]
# ### Phase 4: Standardizing Biological Tissue Set via Probe Annotation Mapping (GSE90603)

from io import StringIO

# 1. Define folder and file paths
gse90603_folder = os.path.join(RAW_DATA_DIR, "4_biological_GSE90603")
matrix_path = os.path.join(gse90603_folder, "GSE90603_series_matrix.txt")
anno_path = os.path.join(gse90603_folder, "GPL21572-124634.txt")

print("⏳ Step 1: Parsing GSE90603 Series Matrix...")
# Filter out metadata lines starting with '!' natively
valid_lines = []
with open(matrix_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.startswith("!") and line.strip():
            valid_lines.append(line)

df_bio = pd.read_csv(StringIO("".join(valid_lines)), sep="\t", index_col=0)
df_bio.index = df_bio.index.astype(str)

print("⏳ Step 2: Loading GPL21572 Translation Map...")
# Load annotation file, ignoring comment lines starting with '#'
df_anno = pd.read_csv(anno_path, sep="\t", comment="#", low_memory=False)

# Build the conversion dictionary: {Probe ID -> Standard miRNA Name}
mapping_dict = dict(zip(df_anno['ID'].astype(str), df_anno['miRNA_ID'].astype(str)))

print("⏳ Step 3: Translating Probe IDs to microRNA Names...")
# Map the matrix index using our dictionary
df_bio.index = df_bio.index.map(mapping_dict)

# Strip out empty probes, control probes, or failed translations (NaNs)
df_bio = df_bio[df_bio.index.notnull()]
df_bio = df_bio[(df_bio.index != 'nan') & (df_bio.index != '')]

# Keep ONLY the genuine human microRNAs (starting with 'hsa-')
df_bio = df_bio[df_bio.index.str.startswith('hsa-', na=False)]

print("⏳ Step 4: Collapsing duplicate probes (Averaging values)...")
# Multiple hardware spots can target the same miRNA; collapse them by taking the mean
df_bio = df_bio.groupby(df_bio.index).mean()
df_bio.index.name = "miRNA"

# 5. Save the translated, polished matrix
output_path_bio = os.path.join(PROCESSED_DATA_DIR, "GSE90603_expression_clean.csv")
df_bio.to_csv(output_path_bio)

print("✅ GSE90603 Probe Translation & Processing Complete!")
print(f"   -> Final Shape (miRNAs x Samples): {df_bio.shape}")
# Look closely at these to confirm the numbers became real 'hsa-' names!
print(f"   -> Translated First 3 miRNAs: {list(df_bio.index[:3])}") 
print(f"   -> Saved to: {output_path_bio}\n")

# Preview the top-left section of our newly translated table
df_bio.iloc[:5, :4]

⏳ Step 1: Parsing GSE90603 Series Matrix...
⏳ Step 2: Loading GPL21572 Translation Map...
⏳ Step 3: Translating Probe IDs to microRNA Names...
⏳ Step 4: Collapsing duplicate probes (Averaging values)...
✅ GSE90603 Probe Translation & Processing Complete!
   -> Final Shape (miRNAs x Samples): (4422, 25)
   -> Translated First 3 miRNAs: ['hsa-let-7a-1', 'hsa-let-7a-2', 'hsa-let-7a-2-3p']
   -> Saved to: data/processed\GSE90603_expression_clean.csv



,GSM2407549,GSM2407550,GSM2407551,GSM2407552
miRNA,,,,
hsa-let-7a-1,1.341710,1.442925,2.306165,2.291760
hsa-let-7a-2,0.667280,0.859517,0.616050,1.586060
hsa-let-7a-2-3p,1.120140,1.588910,0.727678,1.094640
hsa-let-7a-3,1.054640,1.105060,1.173810,1.086990
hsa-let-7a-3p,0.657721,0.822272,1.523590,0.714111


In [9]:
# %% [markdown]
# ### Phase 5: Standardizing External Validation Set (CGGA)

# 1. Define file path 
# Note: Check your 'data' folder to ensure this matches your actual CGGA expression filename exactly!
cgga_folder = os.path.join(RAW_DATA_DIR, "3_external_CGGA198")

# Let's dynamically find the expression file in the CGGA directory
cgga_file = None
if os.path.exists(cgga_folder):
    for f in os.listdir(cgga_folder):
        if "matrix" in f.lower() or "expr" in f.lower() or f.endswith(".txt") or f.endswith(".csv"):
            # We skip clinical files if they are in the same folder
            if "clinical" not in f.lower():
                cgga_file = os.path.join(cgga_folder, f)
                break

print(f"⏳ Loading and cleaning CGGA dataset from: {os.path.basename(cgga_file) if cgga_file else 'Not Found'}...")

if not cgga_file:
    print("❌ Error: Could not automatically locate the CGGA expression file.")
    print("Please manually update the 'cgga_file' variable with your exact file path!")
else:
    # 2. Load the CGGA dataset (detect if it's tab or comma separated)
    sep = "\t" if cgga_file.endswith(".txt") else ","
    df_cgga = pd.read_csv(cgga_file, sep=sep, index_col=0)
    
    # Cast index to string for safety
    df_cgga.index = df_cgga.index.astype(str)
    
    # 3. Clean and isolate genuine human miRNAs
    # Ensure any trailing spaces are stripped
    df_cgga.index = df_cgga.index.str.strip()
    
    # Keep ONLY rows starting with 'hsa-'
    df_cgga = df_cgga[df_cgga.index.str.startswith('hsa-', na=False)]
    
    # Collapse any unexpected duplicate rows by taking the mean
    df_cgga = df_cgga.groupby(df_cgga.index).mean()
    df_cgga.index.name = "miRNA"
    
    # 4. Save the polished CGGA matrix
    output_path_cgga = os.path.join(PROCESSED_DATA_DIR, "CGGA_expression_clean.csv")
    df_cgga.to_csv(output_path_cgga)
    
    print("✅ CGGA External Validation Processing Complete!")
    print(f"   -> Final Shape (miRNAs x Samples): {df_cgga.shape}")
    print(f"   -> First 3 verified miRNAs: {list(df_cgga.index[:3])}")
    print(f"   -> Saved to: {output_path_cgga}\n")
    
    # Preview the top-left section of the finalized CGGA table
    df_cgga.iloc[:5, :4]

⏳ Loading and cleaning CGGA dataset from: CGGA.microRNA_array_198_gene_level.20200506.txt...
✅ CGGA External Validation Processing Complete!
   -> Final Shape (miRNAs x Samples): (829, 198)
   -> First 3 verified miRNAs: ['hsa-let-7a', 'hsa-let-7a#', 'hsa-let-7b']
   -> Saved to: data/processed\CGGA_expression_clean.csv



In [10]:
# %% [markdown]
# ### Phase 6: Extracting Binary Labels for Training and Independent Validation Sets

print("⏳ Mapping clinical labels for GSE122488 and GSE184472...")

# ----------------- DATASET 1: GSE122488 -----------------
# Get all sample columns from our cleaned training dataframe
train_samples = df_train.columns

# Generate labels: 0 for Healthy Controls (HC), 1 for Glioma/GBM classes
# Let's see what unique prefixes we have in our headers
train_labels = []
for sample in train_samples:
    if sample.startswith("HC"):
        train_labels.append(0)  # Control
    else:
        train_labels.append(1)  # Tumour / Glioma

df_labels_train = pd.DataFrame({"Sample_ID": train_samples, "Label": train_labels})
output_labels_train = os.path.join(PROCESSED_DATA_DIR, "GSE122488_labels.csv")
df_labels_train.to_csv(output_labels_train, index=False)


# ----------------- DATASET 2: GSE184472 -----------------
# Get all sample columns from our cleaned validation dataframe
val_samples = df_val.columns

# Generate labels: 0 for Normal, 1 for GBM
val_labels = [0 if sample.startswith("Normal") else 1 for sample in val_samples]

df_labels_val = pd.DataFrame({"Sample_ID": val_samples, "Label": val_labels})
output_labels_val = os.path.join(PROCESSED_DATA_DIR, "GSE184472_labels.csv")
df_labels_val.to_csv(output_labels_val, index=False)


print("✅ Labels extracted and saved successfully!")
print(f"   -> GSE122488 Train Class Breakdown:\n{df_labels_train['Label'].value_counts()}")
print(f"   -> GSE184472 Validation Class Breakdown:\n{df_labels_val['Label'].value_counts()}\n")

⏳ Mapping clinical labels for GSE122488 and GSE184472...
✅ Labels extracted and saved successfully!
   -> GSE122488 Train Class Breakdown:
Label
1    22
0    16
Name: count, dtype: int64
   -> GSE184472 Validation Class Breakdown:
Label
0    73
1    45
Name: count, dtype: int64



In [15]:
# %% [markdown]
# ### Phase 7: Extracting Clinical Labels for GSE90603 and CGGA (Corrected)

print("⏳ Step 1: Processing GSE90603 Biological Tissue labels...")

gse90603_matrix_path = os.path.join(RAW_DATA_DIR, "4_biological_GSE90603", "GSE90603_series_matrix.txt")

sample_ids = []
sample_titles = []

# Safely extract the exact lines using basic string splitting
with open(gse90603_matrix_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("!Sample_geo_accession"):
            # Split by tab, skip the header, and strip out the literal quote marks
            sample_ids = [s.replace('"', '').strip() for s in line.strip().split("\t")[1:]]
        elif line.startswith("!Sample_title"):
            sample_titles = [t.replace('"', '').strip().lower() for t in line.strip().split("\t")[1:]]

title_dict = dict(zip(sample_ids, sample_titles))

valid_bio_samples = list(df_bio.columns)
bio_labels = []
filtered_bio_samples = []

for sample in valid_bio_samples:
    if sample in title_dict:
        title = title_dict[sample]
        # Match the exact words we found in the diagnostic
        if "glioblastoma" in title or "gbm" in title:
            bio_labels.append(1)  # Glioblastoma
            filtered_bio_samples.append(sample)
        elif "astrocyte" in title or "normal" in title or "healthy" in title:
            bio_labels.append(0)  # Control / Normal cells
            filtered_bio_samples.append(sample)

df_labels_bio = pd.DataFrame({"Sample_ID": filtered_bio_samples, "Label": bio_labels})
output_labels_bio = os.path.join(PROCESSED_DATA_DIR, "GSE90603_labels.csv")
df_labels_bio.to_csv(output_labels_bio, index=False)

# Re-save the matrix to match ONLY these finalized, clinical-matched samples
df_bio_filtered = df_bio[filtered_bio_samples]
df_bio_filtered.to_csv(os.path.join(PROCESSED_DATA_DIR, "GSE90603_expression_clean.csv"))

print(f"✅ GSE90603 Labels Extracted!")
print(f"   -> Breakdown:\n{df_labels_bio['Label'].value_counts()}")


# ----------------- DATASET 4: CGGA CLINICAL COHORT -----------------
print("\n⏳ Step 2: Processing CGGA Clinical Cohort using 'Histology' column...")

# Locate the CGGA clinical file again
cgga_folder = os.path.join(RAW_DATA_DIR, "3_external_CGGA198")
cgga_clinical_file = [os.path.join(cgga_folder, f) for f in os.listdir(cgga_folder) if "clinical" in f.lower()][0]

sep = "\t" if cgga_clinical_file.endswith(".txt") else ","
df_cgga_clin = pd.read_csv(cgga_clinical_file, sep=sep)
df_cgga_clin.columns = df_cgga_clin.columns.str.strip()

# Target the specific columns we found
id_col = "CGGA_ID"
hist_col = "Histology"

# Map 'GBM' to 1, everything else (A, AA, O, OA, etc.) to 0
df_cgga_clin['Label'] = df_cgga_clin[hist_col].apply(lambda x: 1 if str(x).strip().upper() == "GBM" else 0)

# Line up directly with our parsed expression dataset columns
valid_cgga_samples = list(df_cgga.columns)
df_cgga_labels_matched = df_cgga_clin[df_cgga_clin[id_col].astype(str).isin(valid_cgga_samples)].copy()

# Save finalized CGGA labels
df_labels_cgga_final = df_cgga_labels_matched[[id_col, 'Label']].rename(columns={id_col: "Sample_ID"})
output_labels_cgga = os.path.join(PROCESSED_DATA_DIR, "CGGA_labels.csv")
df_labels_cgga_final.to_csv(output_labels_cgga, index=False)

print(f"✅ CGGA Labels Extracted successfully!")
print(f"   -> Breakdown:\n{df_labels_cgga_final['Label'].value_counts()}")

⏳ Step 1: Processing GSE90603 Biological Tissue labels...
✅ GSE90603 Labels Extracted!
   -> Breakdown:
Label
1    20
0     5
Name: count, dtype: int64

⏳ Step 2: Processing CGGA Clinical Cohort using 'Histology' column...
✅ CGGA Labels Extracted successfully!
   -> Breakdown:
Label
0    117
1     81
Name: count, dtype: int64


In [16]:
# %% [markdown]
# ### Phase 8: Feature Intersection and ML Matrix Formatting

print("⏳ Finding the universal microRNA intersection...")

# 1. Load the finalized, clean expression datasets
df_tr = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "GSE122488_expression_clean.csv"), index_col=0)
df_va = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "GSE184472_expression_clean.csv"), index_col=0)
df_cg = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "CGGA_expression_clean.csv"), index_col=0)
df_bi = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "GSE90603_expression_clean.csv"), index_col=0)

# 2. Find the overlapping features (intersection of all indices)
common_mirnas = set(df_tr.index) & set(df_va.index) & set(df_cg.index) & set(df_bi.index)
common_mirnas = sorted(list(common_mirnas)) # Sort alphabetically for consistency

print(f"✅ Discovered {len(common_mirnas)} universal microRNAs present across ALL 4 datasets!")

# 3. Filter all datasets to only these common miRNAs and Transpose
X_train = df_tr.loc[common_mirnas].T
X_val   = df_va.loc[common_mirnas].T
X_cgga  = df_cg.loc[common_mirnas].T
X_bio   = df_bi.loc[common_mirnas].T

# 4. Load the labels and align the order perfectly
def align_labels(X_matrix, label_filename):
    df_labels = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, label_filename))
    # Set the sample ID as the index to match X_matrix
    df_labels.set_index("Sample_ID", inplace=True)
    # Ensure exact same row order as X_matrix
    y_aligned = df_labels.loc[X_matrix.index, "Label"]
    return y_aligned

y_train = align_labels(X_train, "GSE122488_labels.csv")
y_val   = align_labels(X_val, "GSE184472_labels.csv")
y_cgga  = align_labels(X_cgga, "CGGA_labels.csv")
y_bio   = align_labels(X_bio, "GSE90603_labels.csv")

# 5. Save the final, ML-Ready matrices
ML_DIR = os.path.join("data", "ml_ready")
os.makedirs(ML_DIR, exist_ok=True)

# Save X (Features)
X_train.to_csv(os.path.join(ML_DIR, "X_train.csv"))
X_val.to_csv(os.path.join(ML_DIR, "X_val.csv"))
X_cgga.to_csv(os.path.join(ML_DIR, "X_cgga.csv"))
X_bio.to_csv(os.path.join(ML_DIR, "X_bio.csv"))

# Save Y (Labels)
y_train.to_csv(os.path.join(ML_DIR, "y_train.csv"))
y_val.to_csv(os.path.join(ML_DIR, "y_val.csv"))
y_cgga.to_csv(os.path.join(ML_DIR, "y_cgga.csv"))
y_bio.to_csv(os.path.join(ML_DIR, "y_bio.csv"))

print(f"🎯 Final ML-Ready Datasets saved to: {ML_DIR}")
print(f"   -> Train Shape: {X_train.shape}")
print(f"   -> Val Shape:   {X_val.shape}")
print(f"   -> CGGA Shape:  {X_cgga.shape}")
print(f"   -> Bio Shape:   {X_bio.shape}")

⏳ Finding the universal microRNA intersection...
✅ Discovered 201 universal microRNAs present across ALL 4 datasets!
🎯 Final ML-Ready Datasets saved to: data\ml_ready
   -> Train Shape: (38, 201)
   -> Val Shape:   (118, 201)
   -> CGGA Shape:  (198, 201)
   -> Bio Shape:   (25, 201)
